In [0]:
from pyspark.sql import functions as F
from collections import Counter

project_path = "/Volumes/workspace/default/freddie_mac_raw/DB Project/"

# Find this project's files.
def find_text_files(folder):
    results = []
    for item in dbutils.fs.ls(folder):
        if item.isDir():
            results.extend(find_text_files(item.path))
        elif item.name.endswith(".txt"):
            results.append(item)
    return results

project_files = find_text_files(project_path)

expected_files = {f"sample_{kind}_{year}.txt"
    for year in range(2016, 2026)
    for kind in ["orig", "perf"]
}

assert Counter(f.name for f in project_files) == Counter(expected_files), \
    "Expected exactly one of each 2016–2025 origination/performance file."
assert all(f.size > 0 for f in project_files), "An input file is empty."

# Column names in the exact order of the uploaded files.
orig_columns = [
    "fico", "first_payment_date", "first_time_homebuyer",
    "maturity_date", "msa", "mi_pct", "units", "occupancy",
    "cltv", "dti", "original_upb", "ltv", "original_rate",
    "channel", "prepayment_penalty", "amortization_type",
    "state", "property_type", "postal_code", "loan_id",
    "purpose", "original_term", "borrowers", "seller",
    "super_conforming", "pre_harp_id", "eligibility_program",
    "harp", "valuation_method", "interest_only", "vantage_score"
]

perf_columns = [
    "loan_id", "period", "current_upb", "delinquency",
    "loan_age", "remaining_term", "defect_date", "modification",
    "zero_balance_code", "zero_balance_date", "current_rate",
    "deferred_upb", "ddlpi", "mi_recoveries", "net_sale_proceeds",
    "non_mi_recoveries", "expenses", "legal_costs",
    "maintenance_costs", "taxes_insurance",
    "miscellaneous_expenses", "actual_loss",
    "cumulative_modification_cost", "step_indicator",
    "deferral", "estimated_ltv", "removal_upb",
    "delinquent_interest", "disaster", "assistance",
    "current_modification_cost", "interest_bearing_upb",
    "mi_cancellation", "servicer", "bankruptcy_cost"
]

def build_bronze(kind, columns):
    paths = sorted(f.path for f in project_filesif f.name.startswith(f"sample_{kind}_"))

    lines = (
        spark.read.text(paths)
        .select(
            F.col("value").alias("raw_line"),
            F.col("_metadata.file_name").alias("source_file"),
            F.col("_metadata.file_path").alias("source_path")
        )
        .withColumn(
            "_fields",
            F.split(F.col("raw_line"), r"\|", -1)
        )
    )

    # Stop before saving if any row has an unexpected structure.
    invalid = lines.filter(
        (F.size("_fields") != len(columns))
        | (F.trim(F.col("raw_line")) == "")
    )

    assert invalid.limit(1).count() == 0, \
        f"Invalid row structure found in {kind} files."

    return lines.select(
        *[
            F.col("_fields").getItem(i).alias(name)
            for i, name in enumerate(columns)
        ],
        "raw_line",
        "source_file",
        "source_path",
        F.regexp_extract(
            F.col("source_file"), r"_(\d{4})\.txt$", 1
        ).cast("int").alias("source_vintage"),
        F.current_timestamp().alias("ingested_at")
    )

bronze_orig = build_bronze("orig", orig_columns)
bronze_perf = build_bronze("perf", perf_columns)

# Keep this project's tables in a dedicated schema.
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.mortgage_risk")

# Save new Delta tables. Existing tables will raise an error.
bronze_orig.write.format("delta").mode("error").saveAsTable(
    "workspace.mortgage_risk.bronze_origination"
)

bronze_perf.write.format("delta").mode("error").saveAsTable(
    "workspace.mortgage_risk.bronze_performance"
)

print("Both Bronze tables saved successfully.")

Both Bronze tables saved successfully.


In [0]:
display(spark.sql("""
    SELECT
        'origination' AS table_name,
        COUNT(*) AS row_count,
        COUNT(DISTINCT source_file) AS source_files
    FROM workspace.mortgage_risk.bronze_origination

    UNION ALL

    SELECT
        'performance' AS table_name,
        COUNT(*) AS row_count,
        COUNT(DISTINCT source_file) AS source_files
    FROM workspace.mortgage_risk.bronze_performance
"""))

table_name,row_count,source_files
origination,500000,10
performance,20079736,10
